In [ ]:
%pip install --quiet beautifulsoup4 matplotlib pandas requests

# Data Collection and Conversion: From a Messy Inbox to a Tidy Dataset

Welcome to the Unstructured Data module. Before we can analyse text, scrape websites or call APIs, we need to be able to answer a more basic question: **what shape is this data in, and what would it take to make it useful?**

You have just joined the analytics team at **Pinnacle Broadband**, a regional internet provider in the South West of England. The Head of Customer Experience has a simple request:

> *"Tell me what customers have been contacting us about this month - and whether anything is getting worse."*

There is no single table to query. The evidence is spread across four places, each in a different shape: a CRM export, a web-chat log, the support mailbox and a public reviews page. In this notebook you will turn that mixed bag into **one tidy, privacy-safe dataset**, and then pull a first piece of insight out of it.

By the end, you will be able to:

- **Distinguish** structured, semi-structured and unstructured data, and recognise that most real sources are a mixture *(Remember & Understand)*
- **Apply** Python to parse CSV, JSON, email and HTML into a common schema *(Apply)*
- **Analyse** the problems each format hides - ambiguous dates, nested records, quoted replies, page clutter *(Analyse)*
- **Evaluate** what the free text tells you that the structured fields do not *(Evaluate)*
- **Create** a reusable, documented pipeline that pseudonymises and redacts personal data before saving *(Create)*

```
Raw sources → Inspect & classify → Parse each format → Common schema → Protect → Combine & save → First insight
```

## How this notebook works

| Marker | Meaning |
|---|---|
| 🏫 | **In-lesson** - we work through this together in the live session |
| 📖 | **Independent study** - complete after the session |
| 📝 | **Exercises** - practise and extend what you have learned |

Two kinds of callout appear throughout:

> 📌 **Link to your assessment** - how this step connects to the summative brief and its marking criteria.

> 🔍 **Design decision** - a choice we made, and why. These are the decisions your technical report (Section C) should explain in your own work.

Run the cells from top to bottom. Everything up to Part 5 works **offline** - the data is created by the notebook itself.

---
## Setup 🏫

All imports and settings live in one place, so you can see at a glance what the notebook depends on and change a setting without hunting for it.

In [ ]:
import email
import hashlib
import json
import re
from email import policy
from email.utils import parsedate_to_datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from bs4 import BeautifulSoup, Tag

# -- Session configuration ----------------------------------------------------
DATA_DIR = Path("pinnacle_data")      # everything this notebook writes goes here
LOCAL_TZ = "Europe/London"            # the timezone Pinnacle's CRM records times in
PSEUDONYM_SALT = "topic1-demo-salt"   # see the 🔍 note in Part 3 - never hard-code a real one

# Module chart palette
NAVY, STEEL, AMBER, GREY = "#1B3A5C", "#2E6DA4", "#D4820A", "#8C99A6"

pd.set_option("display.max_colwidth", 90)
print("Setup complete.")

### Create the Pinnacle inbox

The next cell writes the four raw files we will work with into `pinnacle_data/inbox/`. Its code is **collapsed** because it is just data - you do not need to read it. Run it, then look at the file listing it prints.

Everything in it is fictional: the people, account numbers and addresses are invented, email domains use the reserved `example.*` names, and phone numbers use Ofcom's range set aside for drama.

In [ ]:
# -- Setup: build the Pinnacle Broadband "inbox" -----------------------------
# You do not need to read this cell. It writes the four raw source files that
# the rest of the notebook works with, so the lesson runs identically offline,
# behind a proxy, and for every learner at once.
# All people, accounts, emails and phone numbers are fictional. Email domains
# use the reserved example.* names and phone numbers use Ofcom's drama range.

from pathlib import Path
import json

INBOX = DATA_DIR / "inbox"
(INBOX / "emails").mkdir(parents=True, exist_ok=True)

# 1. Support-ticket export from the CRM (CSV) ---------------------------------
TICKETS_CSV = """ticket_id,opened,account_no,postcode,category,priority,agent_notes
T-10231,01/09/2026 09:12,PB-4001187,EX4 6,Billing,Low,"Cust queried Sept bill - price rise not expected. Explained annual increase, cust unhappy, mentioned switching provider"
T-10232,01/09/2026 10:47,PB-4003321,BS3 4,Billing,Medium,"Charged twice for Aug. Refund raised £32.00, ref RF-8812"
T-10233,02/09/2026 14:05,PB-4002290,PL25 3,Technical,Medium,"PB-500 wifi drops every evening since firmware 2.4 update. Factory reset done, no change. Sent replacement router"
T-10234,03/09/2026 08:30,PB-4005512,TR1 2,Installation,High,"Engineer no-show for install appt 02/09. 2nd missed appt. Rebooked 09/09, cust v. frustrated"
T-10235,03/09/2026 16:22,PB-4001902,TQ2 5,Technical,Low,"Slow speeds evenings ~8Mbps on 150 package. Line test OK. Advised wired test"
T-10236,04/09/2026 11:40,PB-4006630,BA2 3,General,Low,"Wants to upgrade to 500Mb. Quoted, cust to think about it"
T-10237,05/09/2026 19:02,PB-4002771,PL1 3,Technical,Medium,"wifi keeps dropping on PB-500, 2.4 firmware again?? escalated to 2nd line"
T-10238,07/09/2026 09:55,PB-4004410,EX39 1,Billing,Medium,"Price rise letter not received. Cust says contract is fixed price. Wants to cancel w/o fee"
T-10239,08/09/2026 13:18,PB-4003008,TR15 2,Technical,Medium,"Buffering / slow speeds, 3 kids gaming. Suggested upgrade"
T-10240,09/09/2026 10:01,PB-4005512,TR1 2,Installation,Medium,"Install completed today. Engineer replaced long drop wire"
T-10241,10/09/2026 15:44,PB-4007124,TA1 4,General,Low,"Moving house end of month, wants to take service. Moving form sent"
T-10242,11/09/2026 12:12,PB-4002290,PL25 3,Technical,Low,"Replacement PB-500 arrived, same wifi drop issue. Firmware 2.4 known issue? raised with NOC"
T-10243,14/09/2026 07:48,PB-4008801,TR18 4,Technical,High,"No service since approx 03:00. Storm damage reported in area. Linked to INC-5521"
T-10244,14/09/2026 08:15,PB-4003008,TR15 2,Technical,High,"Total loss of service, no sync on router. INC-5521"
T-10245,14/09/2026 08:20,PB-4009910,TR18 4,General,High,"Cust works from home, no internet, asking for compensation. Advised auto-comp once restored"
T-10246,14/09/2026 09:02,PB-4001550,PL27 7,Technical,High,"Line down since early hours. Storm. INC-5521"
T-10247,14/09/2026 10:31,PB-4006002,TR1 2,General,Medium,"Called re outage, wants ETA. Gave 24-48h estimate"
T-10248,14/09/2026 11:50,PB-4007788,PL27 7,Technical,High,"no connection, lights flashing red. Area outage INC-5521"
T-10249,15/09/2026 08:05,PB-4008801,TR18 4,Technical,High,"Still no service, 2nd day. Cust on priority services register - flagged"
T-10250,15/09/2026 09:40,PB-4004455,PL25 3,General,Medium,"Outage in area? Neighbours on other ISPs also down. Confirmed INC-5521"
T-10251,15/09/2026 13:13,PB-4002771,PL1 3,Technical,Low,"Wifi drops resolved after firmware 2.5 pushed. Closing"
T-10252,16/09/2026 09:27,PB-4001550,PL27 7,Technical,Medium,"Service restored overnight but speeds slow, 20Mbps on 150. Monitor 48h"
T-10253,16/09/2026 10:00,PB-4009910,TR18 4,Billing,Medium,"Compensation query following outage. Auto-comp applied to next bill"
T-10254,17/09/2026 14:36,PB-4003321,BS3 4,Billing,Low,"Refund RF-8812 confirmed received. Cust happy"
T-10255,18/09/2026 16:50,PB-4005020,EX4 6,General,Low,
T-10256,21/09/2026 11:11,PB-4004410,EX39 1,General,High,"Cancellation request - price rise. Retention offer declined. Cancel processed, ends 21/10"
T-10257,22/09/2026 09:45,PB-4007124,TA1 4,Installation,Medium,"Home move install booked 30/09"
T-10258,23/09/2026 18:20,PB-4001902,TQ2 5,Technical,Low,"slow speeds again evenings. Wired test 140Mbps so wifi issue - PB-500 placement advice"
T-10259,24/09/2026 12:02,PB-4006630,BA2 3,Billing,Low,"Upgrade to 500Mb applied, new DD amount confirmed"
T-10260,25/09/2026 10:34,PB-4002002,PL1 3,Technical,Medium,"PB-500 on firmware 2.5 - wifi drops again, customer says 'worse than before'"
"""
(INBOX / "tickets.csv").write_text(TICKETS_CSV, encoding="utf-8")

# 2. Web-chat export (JSON) ---------------------------------------------------
CHATS = [
    {"conversation_id": "CH-7701", "started_at": "2026-09-02T18:41:07Z",
     "customer": {"account_no": "PB-4002771", "postcode": "PL1 3"}, "satisfaction": 2,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-02T18:41:07Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-02T18:41:40Z", "text": "my wifi keeps cutting out every evening, the router is the PB-500"},
        {"sender": "bot", "sent_at": "2026-09-02T18:41:42Z", "text": "Sorry to hear that. Have you tried restarting your router?"},
        {"sender": "customer", "sent_at": "2026-09-02T18:42:15Z", "text": "yes about 5 times!! it started after it updated itself last week"},
        {"sender": "agent", "sent_at": "2026-09-02T18:46:30Z", "text": "Thanks for waiting. I can see firmware 2.4 was installed on 28 August. We're aware of wifi drops on some PB-500s and a fix is coming."},
        {"sender": "customer", "sent_at": "2026-09-02T18:47:02Z", "text": "ok but I need it working, can you call me on 07700 900418"}]},
    {"conversation_id": "CH-7702", "started_at": "2026-09-06T10:15:55Z",
     "customer": {"account_no": "PB-4001187", "postcode": "EX4 6"}, "satisfaction": 1,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-06T10:15:55Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-06T10:16:20Z", "text": "Why has my bill gone up £4 a month? Nobody told me about a price rise."},
        {"sender": "agent", "sent_at": "2026-09-06T10:19:48Z", "text": "Our annual price change applied from 1 September and was set out in your contract. I'm sorry it came as a surprise."},
        {"sender": "customer", "sent_at": "2026-09-06T10:20:31Z", "text": "That's not good enough. I'm looking at other providers."}]},
    {"conversation_id": "CH-7703", "started_at": "2026-09-14T06:58:12Z",
     "customer": {"account_no": "PB-4007788", "postcode": "PL27 7"}, "satisfaction": None,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-14T06:58:12Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-14T06:58:40Z", "text": "internet is down, router lights flashing red"},
        {"sender": "bot", "sent_at": "2026-09-14T06:58:41Z", "text": "We're aware of a service outage in your area following last night's storm. Engineers are working to restore service."},
        {"sender": "customer", "sent_at": "2026-09-14T06:59:05Z", "text": "any idea how long? I work from home"}]},
    {"conversation_id": "CH-7704", "started_at": "2026-09-14T08:03:29Z",
     "customer": {"account_no": "PB-4006002", "postcode": "TR1 2"}, "satisfaction": 3,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-14T08:03:29Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-14T08:03:51Z", "text": "No internet since the storm. Is there an outage in Truro?"},
        {"sender": "agent", "sent_at": "2026-09-14T08:07:10Z", "text": "Yes, there's a confirmed outage affecting parts of Cornwall. We expect service back within 24 to 48 hours."},
        {"sender": "customer", "sent_at": "2026-09-14T08:07:44Z", "text": "ok thanks for being upfront"}]},
    {"conversation_id": "CH-7705", "started_at": "2026-09-14T12:22:03Z",
     "customer": {"account_no": "PB-4003008", "postcode": "TR15 2"}, "satisfaction": 1,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-14T12:22:03Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-14T12:22:30Z", "text": "STILL no service. This is the third time I've been in touch today."},
        {"sender": "customer", "sent_at": "2026-09-14T12:22:58Z", "text": "email me updates at kerensa.t@example.co.uk because the phone line is down too"}]},
    {"conversation_id": "CH-7706", "started_at": "2026-09-19T20:10:44Z",
     "customer": {"account_no": "PB-4001902", "postcode": "TQ2 5"}, "satisfaction": 4,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-19T20:10:44Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-19T20:11:10Z", "text": "speeds are really slow tonight, streaming keeps buffering"},
        {"sender": "agent", "sent_at": "2026-09-19T20:14:02Z", "text": "Your line looks healthy. Could you try a device plugged in with a cable? Wifi placement makes a big difference."},
        {"sender": "customer", "sent_at": "2026-09-19T20:25:37Z", "text": "wired is fine, so it's the wifi. I'll move the router out of the cupboard"}]},
    {"conversation_id": "CH-7707", "started_at": "2026-09-22T09:02:18Z",
     "customer": {"account_no": "PB-4007124", "postcode": "TA1 4"}, "satisfaction": 5,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-22T09:02:18Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-22T09:02:45Z", "text": "I'm moving house, can I book the install at the new address?"},
        {"sender": "agent", "sent_at": "2026-09-22T09:05:30Z", "text": "Of course. I've booked an engineer for 30 September, 8am to 1pm."},
        {"sender": "customer", "sent_at": "2026-09-22T09:05:59Z", "text": "brilliant, that was easy"}]},
    {"conversation_id": "CH-7708", "started_at": "2026-09-26T17:48:50Z",
     "customer": {"account_no": "PB-4002002", "postcode": "PL1 3"}, "satisfaction": 2,
     "messages": [
        {"sender": "bot", "sent_at": "2026-09-26T17:48:50Z", "text": "Hi, I'm Pip, Pinnacle's assistant. What can I help with today?"},
        {"sender": "customer", "sent_at": "2026-09-26T17:49:21Z", "text": "the new router firmware has made the wifi drops worse not better"},
        {"sender": "agent", "sent_at": "2026-09-26T17:53:12Z", "text": "I'm sorry. I've added your details to our open issue with the PB-500 team."}]},
]
(INBOX / "chat_export.json").write_text(json.dumps(CHATS, indent=2, ensure_ascii=False), encoding="utf-8")

# 3. Emails to the support mailbox (one .eml file per message) --------------
EMAILS = {
"0001.eml": """From: Morwenna Hicks <morwenna.hicks@example.co.uk>
To: support@pinnacle-broadband.example
Subject: No broadband since the storm
Date: Mon, 14 Sep 2026 07:12:44 +0100
Message-ID: <0001@example.co.uk>
Content-Type: text/plain; charset="utf-8"

Hello,

We have had no broadband at all since about 3am. The router lights are red.
I think the whole street is off - the storm brought a tree down on the lines.
My account number is PB-4008801.

Please can someone let me know when it will be back.

Thanks,
Morwenna

--
Morwenna Hicks
07700 900261
""",
"0002.eml": """From: "Dev Patel" <dev.patel@example.com>
To: support@pinnacle-broadband.example
Subject: Re: Your recent support request [T-10233]
Date: Thu, 10 Sep 2026 21:03:10 +0100
Message-ID: <0002@example.com>
Content-Type: text/plain; charset="utf-8"

The replacement router has exactly the same problem. Wifi drops out every
evening between 7 and 10pm. Account PB 4002290.

On Wed, 2 Sep 2026 at 14:10, Pinnacle Support wrote:
> Thank you for contacting Pinnacle Broadband. We have dispatched a
> replacement PB-500 router which should arrive within 3 working days.
> Kind regards, Pinnacle Support

--
Dev
""",
"0003.eml": """From: Alan Trebilcock <alan.t@example.org>
To: support@pinnacle-broadband.example
Subject: Formal complaint - price increase
Date: Tue, 08 Sep 2026 12:30:00 +0100
Message-ID: <0003@example.org>
MIME-Version: 1.0
Content-Type: multipart/alternative; boundary="BOUNDARY42"

--BOUNDARY42
Content-Type: text/plain; charset="utf-8"

Dear Sir or Madam,

I wish to make a formal complaint. My monthly bill has increased without any
notice that I can find. My contract (account PB-4004410) was sold to me as a
fixed price. If this is not resolved I will cancel and move to another provider.

Yours faithfully,
A. Trebilcock

--BOUNDARY42
Content-Type: text/html; charset="utf-8"

<html><body><p>Dear Sir or Madam,</p><p>I wish to make a formal complaint. My monthly bill has increased without any notice that I can find. My contract (account PB-4004410) was sold to me as a fixed price. If this is not resolved I will cancel and move to another provider.</p><p>Yours faithfully,<br>A. Trebilcock</p></body></html>

--BOUNDARY42--
""",
"0004.eml": """From: sam.oduya@example.net
To: support@pinnacle-broadband.example
Subject: engineer didn't turn up AGAIN
Date: Thu, 03 Sep 2026 08:02:51 +0100
Message-ID: <0004@example.net>
Content-Type: text/plain; charset="utf-8"

second time an engineer hasnt turned up for my install. I took the day off
work for this. account no PB-4005512. call me 07700 900377
""",
"0005.eml": """From: Jenna Rowe <jenna.rowe@example.co.uk>
To: support@pinnacle-broadband.example
Subject: Thank you
Date: Wed, 16 Sep 2026 19:45:02 +0100
Message-ID: <0005@example.co.uk>
Content-Type: text/plain; charset="utf-8"

Just wanted to say thank you to the engineers who were out in the rain on
Monday and Tuesday fixing the lines. Service is back and running well.

Jenna
""",
"0006.eml": """From: Priya Nair <priya.nair@example.com>
To: support@pinnacle-broadband.example
Subject: slow speeds
Date: Mon, 21 Sep 2026 22:14:36 +0100
Message-ID: <0006@example.com>
Content-Type: text/plain; charset="utf-8"

Since the outage last week our speed has been awful, 15Mbps at best when we
pay for 150. Account PB-4001550. Is something still broken?

Sent from my phone
""",
"0007.eml": """From: Tom Kellow <tom.kellow@example.org>
To: support@pinnacle-broadband.example
Subject: Compensation for outage?
Date: Thu, 17 Sep 2026 09:20:18 +0100
Message-ID: <0007@example.org>
Content-Type: text/plain; charset="utf-8"

Hi, we were without internet from Monday to Wednesday morning. Am I entitled
to compensation? Account PB-4009910, postcode TR18 4.

Cheers
Tom

--
Tom Kellow | Kellow Joinery
07700 900512 | tom.kellow@example.org
""",
}
for name, raw in EMAILS.items():
    (INBOX / "emails" / name).write_text(raw, encoding="utf-8")

# 4. A saved copy of the public reviews page (HTML) ---------------------------
REVIEWS_HTML = """<!DOCTYPE html>
<html lang="en-GB">
<head><meta charset="utf-8"><title>Pinnacle Broadband reviews | ReviewHub</title>
<script>window.dataLayer = window.dataLayer || [];</script></head>
<body>
<div class="cookie-banner">We use cookies to improve your experience. <button>Accept all</button></div>
<nav><a href="/">ReviewHub</a> &gt; <a href="/broadband">Broadband</a> &gt; Pinnacle Broadband</nav>
<main>
<h1>Pinnacle Broadband</h1>
<p class="summary">Rated <strong>3.1</strong> out of 5 from 214 reviews</p>

<article class="review" data-rating="1">
  <h3 class="review-title">Three days without internet</h3>
  <p class="reviewer">Lowenna P. &middot; St Ives</p>
  <time datetime="2026-09-16">16 September 2026</time>
  <div class="review-body"><p>Storm on Sunday night and we had no service until Wednesday. Updates were vague and the phone line was engaged for hours.</p></div>
</article>

<article class="review" data-rating="5">
  <h3 class="review-title">Great value</h3>
  <p class="reviewer">Chris W. &middot; Bath</p>
  <time datetime="2026-09-12">12 September 2026</time>
  <div class="review-body"><p>Upgraded to 500Mb and it's rock solid. Easy to do online &amp; half the price of my old provider.</p></div>
</article>

<article class="review" data-rating="2">
  <h3 class="review-title">Router is the weak link</h3>
  <p class="reviewer">Dan M. &middot; Plymouth</p>
  <time datetime="2026-09-09">9 September 2026</time>
  <div class="review-body"><p>The line itself is fine but the PB-500 wifi drops out every evening.<br>Support know about it but no fix yet.</p></div>
</article>

<article class="review" data-rating="4">
  <h3 class="review-title">Engineers were brilliant</h3>
  <p class="reviewer">Helen R. &middot; Newquay</p>
  <time datetime="2026-09-17">17 September 2026</time>
  <div class="review-body"><p>Lost service in the storm but the engineers were out in awful weather and had us back online in two days. Credit where it's due 👏</p></div>
</article>

<article class="review" data-rating="1">
  <h3 class="review-title">Sneaky price rise</h3>
  <p class="reviewer">G. Tonkin &middot; Exeter</p>
  <time datetime="2026-09-05">5 September 2026</time>
  <div class="review-body"><p>Bill went up this month with no warning I noticed. Customer service just said "it's in the contract". Looking elsewhere.</p></div>
</article>

<article class="review" data-rating="3">
  <h3 class="review-title">OK once it's installed</h3>
  <p class="reviewer">Sam O. &middot; Truro</p>
  <time datetime="2026-09-11">11 September 2026</time>
  <div class="review-body"><p>Two missed engineer appointments before the install happened. Speeds are good now though.</p></div>
</article>

<article class="review" data-rating="2">
  <h3 class="review-title">Slow in the evenings</h3>
  <p class="reviewer">Anon &middot; Torquay</p>
  <time datetime="2026-09-24">24 September 2026</time>
  <div class="review-body"><p>Evenings are painfully slow on wifi. Wired is fine apparently, but who wires a phone?</p></div>
</article>

<article class="review" data-rating="5">
  <h3 class="review-title">Moving was painless</h3>
  <p class="reviewer">Ruth A. &middot; Taunton</p>
  <time datetime="2026-09-23">23 September 2026</time>
  <div class="review-body"><p>Booked my home move install on the chat in five minutes. Friendly and quick.</p></div>
</article>

<article class="review" data-rating="1">
  <h3 class="review-title">Worse after the update</h3>
  <p class="reviewer">Anon &middot; Plymouth</p>
  <time datetime="2026-09-27">27 September 2026</time>
  <div class="review-body"><p>The router update was meant to fix the wifi dropping. It has made it worse!!</p></div>
</article>

<article class="review" data-rating="4">
  <h3 class="review-title">Reliable, mostly</h3>
  <p class="reviewer">Mike B. &middot; Bristol</p>
  <time datetime="2026-09-18">18 September 2026</time>
  <div class="review-body"><p>Had a double charge in August but the refund came through within a fortnight. Service itself has been reliable.</p></div>
</article>
</main>
<footer><p>&copy; 2026 ReviewHub. <a href="/terms">Terms</a> &middot; <a href="/privacy">Privacy</a></p></footer>
</body>
</html>
"""
(INBOX / "reviews.html").write_text(REVIEWS_HTML, encoding="utf-8")

print(f"Inbox ready at {INBOX.resolve()}")
for path in sorted(INBOX.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(INBOX)!s:<22} {path.stat().st_size:>6,} bytes")

---
## Part 1 - What have we been given? 🏫

### 1.1 Structured, semi-structured and unstructured

| | Structured | Semi-structured | Unstructured |
|---|---|---|---|
| **Shape** | Fixed rows and columns; the schema is decided in advance | Self-describing tags or keys; flexible and often nested | No predefined data model |
| **Examples** | Database tables, CSV exports | JSON, XML, HTML, email headers | Free text, images, audio, scanned letters |
| **To query it** | SQL or pandas, straight away | Parse it first, then query | Process it first (NLP, OCR, speech-to-text) |

You will often read that **80–90% of organisational data is unstructured**. Treat that as an industry estimate that varies widely between organisations, not as a measured fact - and if you use a figure like it in your report, cite where it came from.

The more useful point is that **very little real data is purely one type**. Keep that in mind as we open each file.

### 1.2 Look before you parse

The first rule of working with any new source: **look at the raw file before you write a single line of parsing code.** A helper function saves us repeating ourselves.

In [ ]:
def peek(path: Path, n_chars: int = 600) -> None:
    """Print the first few hundred characters of a file exactly as stored on disk."""
    raw = path.read_text(encoding="utf-8")
    print(f"-- {path.name}  ({len(raw):,} characters)\n")
    print(raw[:n_chars])
    if len(raw) > n_chars:
        print("…")


peek(INBOX / "tickets.csv")

In [ ]:
peek(INBOX / "chat_export.json")

In [ ]:
peek(INBOX / "emails" / "0002.eml")

In [ ]:
peek(INBOX / "reviews.html", n_chars=900)

### 1.3 Classify each source

**Pause before reading on.** For each file, decide: is it structured, semi-structured or unstructured? And which *parts* of it are which?

| Source | Container | What is inside | Classification |
|---|---|---|---|
| `tickets.csv` | Rows and columns | `agent_notes` is free text | **Structured**, with an unstructured column |
| `chat_export.json` | Nested keys and lists | Each message's `text` is free text | **Semi-structured** |
| `emails/*.eml` | Header fields, then a body | Body is free text, plus quoted replies and signatures | **Semi-structured** headers, **unstructured** body |
| `reviews.html` | Markup tags | Review text - surrounded by cookie banners and navigation | **Semi-structured** markup around **unstructured** text |

The **container** tells you how to *read* the file. The **content** tells you how much *work* the text will need. And notice where the business value sits: in every case, the most useful information is in the free-text parts.

### 1.4 Decide the target shape first

Before writing any parsers, we decide what **one row** of our final dataset means and what columns it will have. Here, one row is **one customer contact**, whichever channel it arrived through.

| Column | Meaning |
|---|---|
| `record_id` | The source's own identifier, so every row can be traced back |
| `source` | `ticket`, `chat`, `email` or `review` |
| `received_at` | When the contact happened, in **UTC** |
| `account_no` | Working column only - replaced by a pseudonymous `customer_ref` in Part 3 |
| `postcode_area` | Area letters only (e.g. `TR`) - enough to spot regional patterns |
| `rating` | A 1–5 score where the source has one |
| `text` | What was said |

> 🔍 **Design decision** - Deciding the target schema *before* writing parsers turns four open-ended problems into four small, testable ones: *"map this source onto these seven columns"*. It also forces early decisions about grain (what one row means) and about what we deliberately **will not** keep.

In [ ]:
COMMON_COLUMNS = ["record_id", "source", "received_at", "account_no", "postcode_area", "rating", "text"]


def show_stage(df: pd.DataFrame, label: str, n: int = 3) -> pd.DataFrame:
    """Print a one-line summary of a pipeline stage and return its first rows for display."""
    print(f"Stage: {label} - {len(df)} rows × {df.shape[1]} columns")
    return df.head(n)

> 📌 **Link to your assessment** - Section B asks for **anonymised screenshots of your data at each stage** of processing. `show_stage` builds that habit: a labelled snapshot after every transformation. But look carefully at what the early stages contain - real account numbers, phone numbers and email addresses. In your own submission, only screenshot stages **after** pseudonymisation and redaction (Part 3), or use a synthetic sample. Identifying information in a screenshot is the most serious problem a marker can find.

---
## Part 2 - Converting each source 🏫

We now write one small function per source, each mapping its format onto the common schema.

### 2.1 The CSV export: structured, but not trouble-free

In [ ]:
tickets_raw = pd.read_csv(INBOX / "tickets.csv")
tickets_raw.info()

pandas has read the file, but look at the `Dtype` column: `opened` is stored as text (`object` or `str`), not as a date. Let's convert it - and see why *"let pandas work it out"* is a trap.

In [ ]:
# ⚠️ Let pandas guess the format from two of the rows (1 and 3 September)
pd.to_datetime(tickets_raw["opened"].iloc[[0, 3]])

The CRM writes **UK dates** (`01/09/2026` is 1 September), but pandas guessed the **US** convention and produced **9 January** and **9 March**. No error, no warning - just wrong data. On the full column it would eventually hit `14/09/2026`, which cannot be a US date, and fail; a silent error in a smaller extract is far more dangerous.

The fix is to state the format explicitly. We also record *which timezone* the times are in, then convert everything to UTC so that sources from different systems can be compared.

In [ ]:
tickets = tickets_raw.copy()
tickets["opened"] = (
    pd.to_datetime(tickets["opened"], format="%d/%m/%Y %H:%M")  # say what the format is
    .dt.tz_localize(LOCAL_TZ)                                   # say which clock it was
    .dt.tz_convert("UTC")                                       # store everything in UTC
)
tickets[["ticket_id", "opened"]].head(3)

> 🔍 **Design decision** - All timestamps are stored in **UTC**. The chat system records UTC, the emails carry a `+0100` offset and the CRM uses UK local time with no offset at all. Converting at the point of parsing means every later comparison ("what happened on the 14th?") is comparing like with like - and it survives the clocks changing in October.

Now we map the tickets onto the common schema. We only need the **area letters** of each postcode, so we write a small helper for that too.

In [ ]:
def postcode_area(postcode: str | None) -> str | None:
    """Reduce a postcode such as 'TR18 4' to its area letters ('TR')."""
    if not isinstance(postcode, str):
        return None
    match = re.match(r"\s*([A-Za-z]{1,2})\d", postcode)
    return match.group(1).upper() if match else None


def tickets_to_records(df: pd.DataFrame) -> pd.DataFrame:
    """Map the CRM ticket export onto the common schema."""
    return pd.DataFrame({
        "record_id": df["ticket_id"],
        "source": "ticket",
        "received_at": df["opened"],
        "account_no": df["account_no"],
        "postcode_area": df["postcode"].map(postcode_area),
        "rating": None,
        "text": df["agent_notes"].fillna(""),
    })[COMMON_COLUMNS]


ticket_records = tickets_to_records(tickets)
show_stage(ticket_records, "tickets → common schema")

> 🔍 **Design decision** - *Whose words are these?* The ticket `text` is the **agent's summary** of the call, full of shorthand ("cust", "appt", "INC-5521"). It is not the customer's voice. We keep it, but the `source` column means anyone analysing sentiment later can separate paraphrase from first-hand text.

### 2.2 The chat export: nested JSON

JSON is **semi-structured**: keys describe the values, but records can nest inside records. Each conversation here contains a list of messages.

In [ ]:
with open(INBOX / "chat_export.json", encoding="utf-8") as f:
    chats = json.load(f)

print(f"{type(chats).__name__} of {len(chats)} conversations")
print(f"Keys in each conversation: {list(chats[0].keys())}")
chats[0]["messages"][:2]

`pd.json_normalize` flattens nested JSON into a table. We ask for **one row per message** (`record_path`) and carry the conversation-level details down to each message (`meta`).

In [ ]:
messages = pd.json_normalize(
    chats,
    record_path="messages",
    meta=["conversation_id", ["customer", "account_no"], ["customer", "postcode"], "satisfaction"],
)
show_stage(messages, "chat messages flattened", n=6)

In [ ]:
messages["sender"].value_counts()

Three different voices share this log: the **customer**, a human **agent** and the **bot**. In conversation `CH-7703`, it is the *bot* that says *"We're aware of a service outage in your area"*. If we counted every message, the bot's scripted replies would inflate our outage figures.

> 🔍 **Design decision** - *Grain and voice.* We keep **only the customer's messages** and join them into **one row per conversation**, matching our "one row = one customer contact" grain. The trade-off: we lose the order of the exchange and the agent's responses, which might matter for a different question (such as how quickly issues are resolved).

In [ ]:
def chats_to_records(messages: pd.DataFrame) -> pd.DataFrame:
    """Collapse chat messages to one row per conversation, keeping only the customer's words."""
    customer = messages[messages["sender"] == "customer"]
    grouped = customer.groupby("conversation_id").agg(
        received_at=("sent_at", "min"),
        account_no=("customer.account_no", "first"),
        postcode=("customer.postcode", "first"),
        rating=("satisfaction", "first"),
        text=("text", " ".join),
    ).reset_index()
    return pd.DataFrame({
        "record_id": grouped["conversation_id"],
        "source": "chat",
        "received_at": pd.to_datetime(grouped["received_at"], utc=True),
        "account_no": grouped["account_no"],
        "postcode_area": grouped["postcode"].map(postcode_area),
        "rating": grouped["rating"],
        "text": grouped["text"],
    })[COMMON_COLUMNS]


chat_records = chats_to_records(messages)
show_stage(chat_records, "chats → common schema")

### 2.3 The support mailbox: email

An email is a **semi-structured** header section (`From`, `Date`, `Subject`…) followed by an **unstructured** body. The format is defined by an internet standard, so Python's built-in `email` library can parse it for us.

**Do not write your own regular expressions for email headers.** Headers can wrap over several lines, be encoded, or arrive in several parts. The standard library already handles all of that.

In [ ]:
raw = (INBOX / "emails" / "0003.eml").read_bytes()
msg = email.message_from_bytes(raw, policy=policy.default)

print("From:   ", msg["From"])
print("Date:   ", msg["Date"])
print("Subject:", msg["Subject"])
print("Parts:  ", [part.get_content_type() for part in msg.walk()])
print()
print(msg.get_body(preferencelist=("plain",)).get_content())

This message arrived as **two versions** of the same text - plain and HTML (`multipart/alternative`). `get_body(preferencelist=("plain",))` picks the plain-text version, so we never have to strip HTML tags out of an email.

The body still needs cleaning. Look back at email `0002` in Part 1: it contains the customer's new message, **a quoted copy of our previous reply** (lines starting `>`), and a **signature** after a `--` line. Only the first of those is the customer speaking now.

In [ ]:
def clean_email_body(body: str) -> str:
    """Remove quoted replies and signature blocks from a plain-text email body."""
    kept = []
    for line in body.splitlines():
        stripped = line.strip()
        if stripped == "--":                           # signature delimiter: stop reading
            break
        if stripped.startswith(">"):                   # quoted text from an earlier message
            continue
        if re.match(r"^On .+ wrote:$", stripped):      # "On <date>, <name> wrote:"
            continue
        if stripped.lower() == "sent from my phone":   # mobile client boilerplate
            continue
        kept.append(stripped)
    return re.sub(r"\s+", " ", " ".join(kept)).strip()


example = email.message_from_bytes((INBOX / "emails" / "0002.eml").read_bytes(), policy=policy.default)
example_body = example.get_body(preferencelist=("plain",)).get_content()

print("BEFORE:\n" + example_body)
print("AFTER:\n" + clean_email_body(example_body))

Emails do not carry an account number in a header - customers type it into the body, and they type it inconsistently (`PB-4002290`, `PB 4002290`). A regular expression finds it and normalises it to one format.

In [ ]:
ACCOUNT_PATTERN = re.compile(r"\bPB[-\s]?(\d{7})\b")


def find_account(text: str) -> str | None:
    """Return the first Pinnacle account number in text, normalised to 'PB-1234567'."""
    match = ACCOUNT_PATTERN.search(text)
    return f"PB-{match.group(1)}" if match else None


def parse_email(path: Path) -> dict:
    """Parse one .eml file into a common-schema record."""
    msg = email.message_from_bytes(path.read_bytes(), policy=policy.default)
    body = clean_email_body(msg.get_body(preferencelist=("plain",)).get_content())
    return {
        "record_id": f"EM-{path.stem}",
        "source": "email",
        "received_at": parsedate_to_datetime(msg["Date"]),
        "account_no": find_account(body),
        "postcode_area": None,
        "rating": None,
        "text": f"{msg['Subject']}. {body}",
    }


def emails_to_records(folder: Path) -> pd.DataFrame:
    """Parse every .eml file in a folder onto the common schema."""
    df = pd.DataFrame([parse_email(path) for path in sorted(folder.glob("*.eml"))])
    df["received_at"] = pd.to_datetime(df["received_at"], utc=True)
    return df[COMMON_COLUMNS]


email_records = emails_to_records(INBOX / "emails")
show_stage(email_records, "emails → common schema", n=7)

> 🔍 **Design decision** - *What we deliberately leave behind.* The sender's name and email address are right there in the `From` header, and we **choose not to keep them**. They are not needed to answer the question we were asked, and **data minimisation** - collecting only what you need - is a principle of UK GDPR. The subject line, on the other hand, is kept as part of the text: *"engineer didn't turn up AGAIN"* carries meaning.

### 2.4 The saved reviews page: HTML

HTML is **semi-structured**: tags and attributes mark up the page, and the text we want sits inside them - alongside a cookie banner, navigation links, scripts and a footer.

We are working from a **saved copy** of the page. Fetching pages live, and the questions about whether you are *allowed* to, are the subject of Topic 3.

In [ ]:
html = (INBOX / "reviews.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

print("Page title:", soup.title.get_text())
articles = soup.select("article.review")
print(f"Found {len(articles)} reviews\n")
print(articles[2].prettify())

Each review follows the same pattern, so we can say exactly where each field lives:

| Field | Where it is | Why this choice |
|---|---|---|
| Rating | the `data-rating` **attribute** of `<article>` | Already a number - no need to count star icons |
| Date | the `datetime` **attribute** of `<time>` | The machine-readable ISO date, not the display text "9 September 2026" |
| Text | `.review-title` and `.review-body` | `get_text(" ")` turns the `<br>` into a space and decodes `&amp;` to `&` |
| Reviewer | `.reviewer` | **Not kept** - a name and a town is personal data we do not need |

In [ ]:
def parse_review(article: Tag, index: int) -> dict:
    """Extract one review <article> into a common-schema record."""
    title = article.select_one(".review-title").get_text(strip=True)
    body = article.select_one(".review-body").get_text(" ", strip=True)
    return {
        "record_id": f"RV-{index:03d}",
        "source": "review",
        "received_at": pd.Timestamp(article.select_one("time")["datetime"], tz="UTC"),
        "account_no": None,
        "postcode_area": None,
        "rating": int(article["data-rating"]),
        "text": f"{title}. {body}",
    }


def reviews_to_records(path: Path) -> pd.DataFrame:
    """Parse a saved reviews page onto the common schema."""
    page = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    records = [parse_review(a, i) for i, a in enumerate(page.select("article.review"), start=1)]
    return pd.DataFrame(records)[COMMON_COLUMNS]


review_records = reviews_to_records(INBOX / "reviews.html")
show_stage(review_records, "reviews → common schema", n=4)

> 🔍 **Design decision** - *Sample, not population.* The page header says **214 reviews**; the saved page holds **10**. Anything we conclude from reviews describes this sample. Saying so in your report is not a weakness - it is exactly the kind of bounded claim that the higher grade bands reward.

---
## Part 3 - Combine, check, protect and save 🏫

### 3.1 Combine

Because every source now has **identical columns**, combining them is a single line.

In [ ]:
combined = pd.concat([ticket_records, chat_records, email_records, review_records], ignore_index=True)
combined["rating"] = combined["rating"].astype("Int64")   # whole numbers that may be missing
combined = combined.sort_values("received_at").reset_index(drop=True)

show_stage(combined, "all sources combined", n=5)

In [ ]:
combined["source"].value_counts()

### 3.2 Check

A pipeline that runs without errors is not the same as a pipeline that produced **correct** data. A few explicit checks catch the problems that would otherwise surface in a chart three weeks later.

In [ ]:
def check_records(df: pd.DataFrame) -> pd.DataFrame:
    """Report basic quality problems and return only the records that have text."""
    empty_text = df["text"].str.strip() == ""
    problems = {
        "missing columns": [c for c in COMMON_COLUMNS if c not in df.columns],
        "duplicate record_id": df.loc[df["record_id"].duplicated(), "record_id"].tolist(),
        "missing timestamp": df.loc[df["received_at"].isna(), "record_id"].tolist(),
        "empty text (removed)": df.loc[empty_text, "record_id"].tolist(),
    }
    for check, found in problems.items():
        print(f"{'✗' if found else '✓'} {check}: {found or 'none'}")
    return df[~empty_text].reset_index(drop=True)


checked = check_records(combined)

One ticket (`T-10255`) had no agent notes at all. Rather than let it vanish silently, the check **reports** it - so the decision to drop it is visible and could be revisited.

### 3.3 Protect

Our combined data contains **personal data**: account numbers, phone numbers and email addresses, some typed straight into the free text. Two techniques reduce the risk:

- **Pseudonymisation** replaces an identifier with a stable reference. The same account always maps to the same `customer_ref`, so we can still count repeat contacts - but the reference cannot be turned back into the account number without the secret *salt*.
- **Redaction** replaces identifiers found inside free text with placeholder tokens such as `[PHONE]`.

In [ ]:
def pseudonymise(account_no: str | None, salt: str = PSEUDONYM_SALT) -> str | None:
    """Replace an account number with a stable, non-reversible reference."""
    if not isinstance(account_no, str):
        return None
    digest = hashlib.sha256(f"{salt}:{account_no}".encode("utf-8")).hexdigest()
    return f"C-{digest[:10]}"


REDACTIONS = {
    "[EMAIL]": re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"),
    "[PHONE]": re.compile(r"\b0\d{4}\s?\d{6}\b"),
    "[ACCOUNT]": ACCOUNT_PATTERN,
}


def redact(text: str) -> str:
    """Replace email addresses, phone numbers and account numbers in text with tokens."""
    for token, pattern in REDACTIONS.items():
        text = pattern.sub(token, text)
    return text


print(pseudonymise("PB-4008801"), "←  same input, same output:", pseudonymise("PB-4008801"))
print(redact("call me on 07700 900418 or email kerensa.t@example.co.uk, acc PB 4003008"))

In [ ]:
protected = checked.copy()
protected["customer_ref"] = protected["account_no"].map(pseudonymise)
protected["text"] = protected["text"].map(redact)
protected = protected.drop(columns="account_no")

FINAL_COLUMNS = ["record_id", "source", "received_at", "customer_ref", "postcode_area", "rating", "text"]
protected = protected[FINAL_COLUMNS]

# Before and after for two records that contained contact details
for record_id in ["CH-7705", "EM-0004"]:
    before = checked.loc[checked["record_id"] == record_id, "text"].item()
    after = protected.loc[protected["record_id"] == record_id, "text"].item()
    print(f"{record_id}\n  before: {before}\n  after:  {after}\n")

In [ ]:
# Automated check: nothing that looks like an identifier should survive
leaks = protected["text"].str.contains(r"@|PB[-\s]?\d{7}|\b07\d{3}\s?\d{6}\b", regex=True)
assert not leaks.any(), protected.loc[leaks, ["record_id", "text"]]
print("✓ No email addresses, account numbers or mobile numbers remain in the text.")

Our automated check passes. But an automated check can only find what it was told to look for. Search for a first name:

In [ ]:
names = protected[protected["text"].str.contains(r"Morwenna|Jenna|\bTom\b", regex=True)]
for record_id, text in zip(names["record_id"], names["text"]):
    print(f"{record_id}: …{text[-70:]}")

The signature was removed, but the sign-offs **above** the `--` line ("Thanks, Morwenna") survived. A regular expression cannot know what a name is. Topic 6 introduces Named Entity Recognition, which helps - but no automated method is perfect, and a **manual check before anything is shared** remains part of the job.

> 🔍 **Design decision** - *The salt.* The salt is hard-coded here so the notebook is reproducible in class. In a real system it would be kept out of the code entirely (an environment variable or a secrets manager). Anyone holding both the salt and a list of account numbers could recreate the mapping - which is why **pseudonymised data is still personal data** under UK GDPR. Only properly anonymised data, from which no one can be re-identified, falls outside it.

> 📌 **Link to your assessment** - Anonymisation appears in the brief itself (Section B), and the top grade band for the scraping criterion asks for *"the legal status of the data being collected"*. Explaining **which** identifiers you removed, **how**, and **what your method would miss** is strong evidence for both.

### 3.4 Save

We save two copies of the final dataset. CSV opens anywhere; **JSON Lines** (one JSON record per line) keeps each field's type and can hold nested data if we add it later.

In [ ]:
PROCESSED = DATA_DIR / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

protected.to_csv(PROCESSED / "customer_contacts.csv", index=False)
protected.to_json(PROCESSED / "customer_contacts.jsonl", orient="records", lines=True, date_format="iso")

for path in sorted(PROCESSED.iterdir()):
    print(f"{path.name:<26} {path.stat().st_size:>6,} bytes")

# Read the CSV back: what happened to the timestamps?
print("\nreceived_at type after a CSV round trip:", pd.read_csv(PROCESSED / "customer_contacts.csv")["received_at"].dtype)

A CSV is just text: our carefully converted timestamps come back as strings. Whoever loads the file next must parse them again (`parse_dates=["received_at"]`). This is worth a sentence in your technical report: a file format is part of your pipeline's design, not an afterthought.

### 3.5 The whole pipeline as one function

Each step so far has been a separate cell so we could inspect it. In practice, you wrap the steps in **one function** that runs from raw inbox to clean records - the same pattern as the reusable `preprocess_text` function in Topic 2.

In [ ]:
def build_contacts_dataset(inbox: Path) -> pd.DataFrame:
    """Collect every source in the inbox, convert to the common schema, check and protect.

    Args:
        inbox: Folder containing tickets.csv, chat_export.json, emails/ and reviews.html.

    Returns:
        One row per customer contact, pseudonymised and redacted, sorted by time.
    """
    tickets = pd.read_csv(inbox / "tickets.csv")
    tickets["opened"] = (
        pd.to_datetime(tickets["opened"], format="%d/%m/%Y %H:%M")
        .dt.tz_localize(LOCAL_TZ).dt.tz_convert("UTC")
    )
    with open(inbox / "chat_export.json", encoding="utf-8") as f:
        messages = pd.json_normalize(
            json.load(f), record_path="messages",
            meta=["conversation_id", ["customer", "account_no"], ["customer", "postcode"], "satisfaction"],
        )

    combined = pd.concat([
        tickets_to_records(tickets),
        chats_to_records(messages),
        emails_to_records(inbox / "emails"),
        reviews_to_records(inbox / "reviews.html"),
    ], ignore_index=True)
    combined["rating"] = combined["rating"].astype("Int64")
    combined = combined.sort_values("received_at").reset_index(drop=True)

    records = check_records(combined)
    records["customer_ref"] = records["account_no"].map(pseudonymise)
    records["text"] = records["text"].map(redact)
    return records[FINAL_COLUMNS]


contacts = build_contacts_dataset(INBOX)
pd.testing.assert_frame_equal(contacts, protected)
print("\n✓ The one-function pipeline reproduces the step-by-step result exactly.")

> 📌 **Link to your assessment** - Section A asks for *"an automated process"*, and the reproducibility criterion (LO5, 10%) rewards code that someone else can run and get the same result. Type hints, docstrings, a single entry point and an explicit check (`assert_frame_equal`) are all visible evidence of that. At the top band, you would also say what is **not** deterministic - here, nothing yet; once Part 5 fetches a live web page, the result depends on when you run it.

---
## Part 4 - A first look at the value in the text 🏫

We now have one tidy table. What can it tell the Head of Customer Experience?

### 4.1 The structured view

The CRM already categorises every ticket. Here is what the structured field alone says:

In [ ]:
tickets["category"].value_counts()

"Technical" is the biggest group - but technical *what*? A router fault, a slow line and a storm-damaged cable all need different responses. The answer is in the text.

### 4.2 What the text says

A deliberately simple approach: a dictionary of **themes**, each defined by a regular expression of words that signal it. Every record is tagged with every theme it mentions.

In [ ]:
THEMES = {
    "outage": r"\b(no (service|internet|broadband|connection)|outage|down|loss of service|no sync|storm)\b",
    "wifi / router": r"\b(wi-?fi|router|pb-500|firmware)\b",
    "speed": r"\b(slow|speeds?|buffering|\d+ ?mbps)\b",
    "billing": r"\b(bill|price|charged?|refund|compensation|auto-comp)\b",
    "installation": r"\b(install\w*|engineers?|appointments?|appt)\b",
    "cancellation": r"\b(cancel\w*|switching|other providers?|another provider|looking elsewhere)\b",
}


def tag_themes(text: str) -> list[str]:
    """Return every theme whose pattern appears in the text."""
    return [theme for theme, pattern in THEMES.items() if re.search(pattern, text, flags=re.IGNORECASE)]


contacts["themes"] = contacts["text"].map(tag_themes)
contacts[["record_id", "source", "text", "themes"]].head(8)

In [ ]:
theme_counts = (
    contacts.explode("themes")
    .dropna(subset=["themes"])
    .pivot_table(index="themes", columns="source", values="record_id", aggfunc="count", fill_value=0)
)
theme_counts = theme_counts.loc[theme_counts.sum(axis=1).sort_values().index]

ax = theme_counts.plot(kind="barh", stacked=True, figsize=(10, 5), color=[NAVY, STEEL, AMBER, GREY])
ax.set_title("Figure 1: Customer contacts by theme and channel, September 2026")
ax.set_xlabel("Number of contacts mentioning the theme")
ax.set_ylabel("")
ax.legend(title="Source", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 4.3 What the category field was hiding

Cross-tabulating the CRM's own category against the themes found in the text shows what the structured field could not.

In [ ]:
ticket_view = contacts[contacts["source"] == "ticket"].merge(
    tickets[["ticket_id", "category"]], left_on="record_id", right_on="ticket_id"
).explode("themes").reset_index(drop=True)
pd.crosstab(ticket_view["category"], ticket_view["themes"])

"Technical" splits into three quite different problems - outages, wifi/router faults and speed - and several **outage** contacts were logged as "General" because the customer was asking for an update or compensation rather than reporting a fault. None of that is visible in the structured column.

### 4.4 Is anything getting worse?

In [ ]:
outage_days = (
    contacts[contacts["themes"].map(lambda t: "outage" in t)]
    .assign(day=lambda d: d["received_at"].dt.tz_convert(LOCAL_TZ).dt.date)
    .groupby("day").size()
)
all_days = pd.date_range("2026-09-01", "2026-09-30", freq="D").date
outage_days = outage_days.reindex(all_days, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 4))
colours = [AMBER if n >= 3 else STEEL for n in outage_days.values]
ax.bar([d.strftime("%d") for d in outage_days.index], outage_days.values, color=colours)
ax.set_title("Figure 2: Contacts mentioning an outage, by day (all channels)")
ax.set_xlabel("Day of September 2026")
ax.set_ylabel("Contacts")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Postcode areas of outage contacts:",
      contacts[contacts["themes"].map(lambda t: "outage" in t)]["postcode_area"].value_counts(dropna=False).to_dict())

### 4.5 From observation to insight

Compare two ways of reporting what we have found:

- **Descriptive:** *"There were 29 tickets, 8 chats, 7 emails and 10 reviews. Outage was a common theme."*
- **Analytical:** *"Outage contacts spike on 14 September and continue until the 17th, concentrated in the TR and PL postcode areas and arriving through every channel - consistent with the storm damage customers describe. Four of the nine outage tickets were logged as 'General' or 'Billing', so the CRM's own categories understate the incident. Separately, complaints of wifi drops on the PB-500 router continue after the firmware 2.5 fix, and three contacts on three different channels say it is now worse - suggesting the fix has not worked."*

The second version says **what**, **where**, **when**, **so what** - and points to a decision.

It also needs its limits stated: 54 records, one month, and keyword rules that miss some contacts and misfile others. Two of the nine "installation" records are customers thanking engineers for storm repairs, and "wifi / router" picks up outage reports that mention the router's red lights. These findings are **indicative**. Topics 2 and 6 give you far better tools than a keyword list; the point today is that even crude processing of the text reveals what the structured fields hide.

> 📌 **Link to your assessment** - The move from descriptive to analytical writing is the single biggest difference between the middle and upper grade bands in Section C. Each claim should be specific, supported by an output (a figure or table you can point to), connected to a business consequence, and honest about its limits.

---
# Independent study 📖

Complete these sections after the live session.

## Part 5 - The same content, two shapes 📖

So far every source has been a file someone handed us. Much of the data you will collect in this module comes from the web - and before scraping any page, there is one question worth asking first: **does the publisher already offer this data in a structured form?**

GOV.UK is a good place to see this. Every page on it is also available as JSON through the **GOV.UK Content API**, and its content is published under the **Open Government Licence**, which explicitly permits reuse with attribution. We make **one** request for each version of one page, identify ourselves with a `User-Agent`, and set a `timeout` so that a slow connection cannot hang the notebook.

> ⚠️ If this section fails with an SSL or certificate error, you are probably behind an organisational proxy. Speak to your tutor rather than searching for a workaround - some common fixes online switch off security checks you should keep on.

In [ ]:
import requests

GOVUK_PATH = "/guidance/project-gigabit-uk-gigabit-programme"
HEADERS = {"User-Agent": "BPP-L6DS-UD-teaching-notebook (educational use)"}


def fetch(url: str) -> requests.Response | None:
    """Make one polite GET request; return None instead of failing if it cannot connect."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return response
    except requests.RequestException as error:
        print(f"Could not fetch {url}\n  → {type(error).__name__}: {error}")
        return None


page = fetch("https://www.gov.uk" + GOVUK_PATH)
if page is not None:
    page_soup = BeautifulSoup(page.text, "html.parser")
    print(f"HTML version: {len(page.text):,} characters of markup")
    print("Title:", page_soup.title.get_text(strip=True))
    print("Every <h2> in the main content:")
    for heading in page_soup.select("main h2"):
        print("  -", heading.get_text(strip=True))

Parsing the HTML works, but look at what came back with the real headings: "Contents", "Sign up for emails or print this page", "Explore the topic" - page furniture, not content. We would need extra rules to filter it out, and those rules would break the next time GOV.UK changed its page design.

Now the same page from the Content API:

In [ ]:
api = fetch("https://www.gov.uk/api/content" + GOVUK_PATH)
if api is not None:
    content = api.json()
    print("Title:        ", content["title"])
    print("Publisher:    ", content["links"]["organisations"][0]["title"])
    print("First published:", content["first_published_at"])
    print("Last updated: ", content["public_updated_at"])
    print("Sections:")
    for header in content["details"]["headers"]:
        print("  -", header["text"])

print("\nContains public sector information licensed under the Open Government Licence v3.0.")

The API returns the page's **own** section list with none of the furniture, plus metadata - publisher, publication and update dates - that we would otherwise have to dig out of the HTML. Same content, two shapes: one we must **parse**, one that arrives **structured**.

| | Parse the HTML page | Use the API |
|---|---|---|
| What you get | Everything on the page, including navigation and banners | The content and its metadata, already structured |
| Breaks when | The site's design changes | The API's version changes (announced, usually) |
| Permission | Check terms and `robots.txt` (Topic 3) | Check terms and any usage limits (Topic 4) |

**Where this goes next.** Topic 3 is web scraping done properly - `robots.txt`, terms of service, error handling and rate limiting. Topic 4 covers APIs, JSON and XML in depth. Today's lesson for both: *check for an API before you scrape*.

## Part 6 - Your organisation's data inventory 📖

Your summative assessment asks you to find a business challenge **in your own organisation** where unstructured data can add value. Start that thinking now.

Copy this table into a markdown cell or your notes and complete it for **three** sources of unstructured or semi-structured data at work:

| Source | Where it lives | Format | Structured / semi / unstructured? | What question could it answer? | Personal data in it? | Who would need to approve access? |
|---|---|---|---|---|---|---|
| *e.g. customer complaint emails* | *Shared mailbox* | *.eml / Outlook* | *Semi (headers) + unstructured (body)* | *Why complaints rose in Q3* | *Yes - names, contact details* | *Data protection officer, team manager* |
| | | | | | | |
| | | | | | | |

Then write **around 500 words** addressing:

1. The benefits of structured data and the challenges of unstructured data, with examples from your inventory.
2. At least **two** ways your organisation could use the untapped value in unstructured data.
3. The tools and techniques you would use to extract one of those sources - where it is, how you would get it out, and what you would need to protect.

Bring this to your coach and to the formative assessment: it is the seed of your summative project.

---
## Summary

| Source | Format | Key challenge | Key tool |
|---|---|---|---|
| CRM tickets | CSV | Ambiguous UK dates; free-text notes are the agent's paraphrase | `pd.read_csv`, `pd.to_datetime(format=…)` |
| Web chat | JSON | Nested records; three voices in one log | `json.load`, `pd.json_normalize` |
| Support mailbox | Email (.eml) | Multipart bodies, quoted replies, signatures | `email` library, `get_body` |
| Reviews page | HTML | Content buried in page furniture | `BeautifulSoup`, CSS selectors |
| All of them | Common schema | Personal data in structured *and* free-text fields | `hashlib`, regular expressions |

### Key takeaways

1. **Most real data is a mixture.** The container tells you how to read it; the free text is usually where the value is.
2. **Decide the target shape first.** A common schema turns an open-ended mess into small, testable conversion steps.
3. **Every format hides a trap.** Silent date errors, bot messages, quoted replies and page furniture all produce wrong answers without raising an error.
4. **Protect before you share.** Pseudonymise, redact, check automatically - and then check by eye, because automated rules miss things.
5. **State your limits.** A small sample and keyword rules give indicative findings, and saying so is a strength.

---
## Exercises 📝

1. **Fill in the gaps.** Emails have no postcode area, but many of those customers also raised a ticket or a chat. Use `customer_ref` to fill in `postcode_area` for emails where another record for the same customer has one. How many gaps can you fill?

2. **Change the grain.** Rewrite `chats_to_records` to keep **one row per customer message** instead of one per conversation. How do the theme counts in Figure 1 change, and which grain better answers the Head of Customer Experience's question?

3. **Tune a theme.** Find every record tagged "installation" and decide which are genuine. Change the pattern to remove the false positives without losing true ones, and re-run Figure 1.

4. **Add a fifth source.** Field engineers file job notes as XML. Parse the snippet below into common-schema records (`source="engineer"`) and add them to the pipeline. *Hint: `BeautifulSoup(xml, "xml")` or Python's `xml.etree.ElementTree` - Topic 4 covers XML properly.*

```xml
<jobs>
  <job id="J-3301" completed="2026-09-15T14:20:00+01:00" account="PB-4001550" postcode="PL27 7">
    <notes>Tree down across overhead cable at pole 14. Replaced 80m drop. Service restored.</notes>
  </job>
  <job id="J-3302" completed="2026-09-29T10:05:00+01:00" account="PB-4002002" postcode="PL1 3">
    <notes>PB-500 swapped for new model at customer request. Wifi stable on test.</notes>
  </job>
</jobs>
```

5. **Stretch - test your parsers.** Write tests for `clean_email_body` and `redact` using edge cases that currently slip through: a phone number with no space, one written `+44 7700 900123`, a full postcode typed into the text (look at `EM-0007`), and a sign-off with no `--` line. Fix the functions until your tests pass.

6. **Stretch - the name problem.** Design a rule to remove sign-off names (the last short line of an email before any signature). Test it on all seven emails. Would you trust it enough to put the output in a Section B screenshot? Justify your answer.

---
## What's next?

In **Topic 2**, we take the `text` column you built today and learn to clean and prepare it properly - tokenisation, stop words and lemmatisation - replacing today's keyword list with a real preprocessing pipeline. Topics 3 and 4 then return to the two collection routes you glimpsed in Part 5: scraping the web responsibly, and working with APIs.